# LoRA fine-tuning on Kaggle

**Before running anything**: Settings (right sidebar) -> Accelerator -> **GPU T4 x2** (or P100/A100 if you have quota) -> **Internet: On** (needed for `pip install`, cloning the repo, and downloading the base model weights).

This notebook:
1. Installs dependencies and clones your repo.
2. Gets the OmniMedVQA data and the Qwen3-VL-4B-Instruct base weights onto Kaggle's local disk (two options for each, pick whichever is faster for you).
3. Runs `src/train_lora.py` (4-bit QLoRA by default) on the fine-tuning dataset split.
4. Sanity-checks the trained adapter with a small `evaluate_omnimed.py` run.
5. (Optional, separate concern) Produces a pre-quantized, smaller copy of the base model for your **submission** zip -- see the last section; this is unrelated to fine-tuning itself and only matters for fitting the 10GB Drive limit.

Fill in `<your-username>/<your-repo>` and the exact OmniMedVQA/model-weights source you're using below before running.

## 0. GPU check

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies

If the repo is private, use `https://<TOKEN>@github.com/<user>/<repo>.git` instead.

In [ ]:
!git clone https://github.com/<your-username>/<your-repo>.git
%cd <your-repo>
!pip install -q -r requirements.txt
!pip install -q bitsandbytes peft

## 2. Get the OmniMedVQA data onto Kaggle's disk

**Kaggle notebooks do not mount Google Drive the way Colab does** -- pick ONE of the two options below.

### Option A (recommended, most reliable): upload OmniMedVQA as a Kaggle Dataset
Do this once, outside the notebook: on your own machine (or wherever you can access the Drive folder), download it locally, then either drag-and-drop it into Kaggle's "New Dataset" web UI, or use the Kaggle CLI (`kaggle datasets create -p <local_folder>`). Once created, click "Add Data" in this notebook and attach it -- it will appear at `/kaggle/input/<your-dataset-name>/`.

Then just set:

In [ ]:
DATA_ROOT = "/kaggle/input/<your-omnimedvqa-dataset-name>"  # Option A: after attaching via "Add Data"

### Option B (faster to set up, less reliable): pull directly from Drive with `gdown`
Only works if the Drive folder/zip is shared as "Anyone with the link can view". If OmniMedVQA is a single folder, zip it in Drive first (gdown handles a shared *file* far more reliably than a shared *folder*), then get its file ID from the share link.

**Skip this cell if you used Option A above.**

In [ ]:
# import gdown
# gdown.download(id="<your-zip-file-id>", output="/kaggle/working/OmniMedVQA.zip", quiet=False)
# !unzip -q /kaggle/working/OmniMedVQA.zip -d /kaggle/working/OmniMedVQA
# DATA_ROOT = "/kaggle/working/OmniMedVQA"  # adjust if the zip's top-level folder name differs

## 3. Get the Qwen3-VL-4B-Instruct base weights

### Option A (recommended -- fastest, no manual upload): download fresh from the Hugging Face Hub
It's a public model, no auth needed. Kaggle's network is usually fast enough that this beats manually moving ~9GB out of Drive.

In [ ]:
from huggingface_hub import snapshot_download
from src import config

snapshot_download(repo_id=config.BASE_MODEL_NAME, local_dir=config.MODEL_PATH)

### Option B: use weights already uploaded as a Kaggle Dataset
If you've already got these in a Kaggle Dataset from a previous session (same idea as Option A in Section 2), skip the cell above and instead point `config.MODEL_PATH` at the attached input:
```python
from src import config
config.MODEL_PATH = "/kaggle/input/<your-weights-dataset-name>"
```
Run this **instead of** the snapshot_download cell, not in addition to it.

## 4. Verify the dataset is actually readable (no model load yet, cheap)

In [ ]:
!python src/evaluate_omnimed.py --list-datasets --data-root "{DATA_ROOT}"

## 5. (Optional) Prepare the external volumetric fine-tuning datasets

Only run this section if you've already downloaded the raw MosMedData / Task01_BrainTumour / BUSI archives somewhere accessible to this Kaggle session (e.g. also attached as Kaggle Datasets or pulled via gdown, same pattern as Section 2). If you haven't, skip straight to Section 6 -- fine-tuning still works fine on the OmniMedVQA-only set below, just without the volumetric-specific examples.

In [ ]:
# RUN_EXTERNAL_VOLUMETRIC = False  # flip to True once the three raw datasets are downloaded and paths below are correct
#
# if RUN_EXTERNAL_VOLUMETRIC:
#     !python src/prepare_external_dataset.py --dataset mosmed    --raw-root /kaggle/input/mosmeddata                --output-root "{DATA_ROOT}"
#     !python src/prepare_external_dataset.py --dataset brainmri  --raw-root /kaggle/input/task01-braintumour         --output-root "{DATA_ROOT}"
#     !python src/prepare_external_dataset.py --dataset busi      --raw-root /kaggle/input/busi-dataset                --output-root "{DATA_ROOT}"
#     !python src/evaluate_omnimed.py --list-datasets --data-root "{DATA_ROOT}"  # should now also show MosMedData_train/_eval etc.

## 6. Run LoRA fine-tuning

Default dataset list below is the OmniMedVQA-only fine-tuning set (disjoint from the evaluation set). If you ran Section 5, uncomment the `MosMedData_train BrainMRI_train BUSI_train` line to include the volumetric examples too.

Writes the adapter straight into `/kaggle/working/weights/lora-medvqa` so it survives as a Kaggle Output. Start with a small `--max-samples` on a first run to confirm the loss is actually decreasing before committing your remaining GPU quota to the full set.

In [ ]:
!python src/train_lora.py \
    --data-root "{DATA_ROOT}" \
    --dataset-names ACRIMA "PALM2019" "COVIDx CXR-4" "Mura" "ISIC2018" "Fitzpatrick 17k" "Covid CT" "BreakHis" "ALL Challenge" "NLM- Malaria Data" "OCT & X-Ray 2017" \
    --max-samples 0 \
    --epochs 3 \
    --grad-accum-steps 8 \
    --output-dir /kaggle/working/weights/lora-medvqa \
    --checkpoint-dir /kaggle/working/lora_training_run

**If you ran Section 5**, use this instead (adds the volumetric training examples):
```bash
!python src/train_lora.py \
    --data-root "{DATA_ROOT}" \
    --dataset-names ACRIMA "PALM2019" "COVIDx CXR-4" "Mura" "ISIC2018" "Fitzpatrick 17k" "Covid CT" "BreakHis" "ALL Challenge" "NLM- Malaria Data" "OCT & X-Ray 2017" MosMedData_train BrainMRI_train BUSI_train \
    --max-samples 0 --epochs 3 --grad-accum-steps 8 \
    --output-dir /kaggle/working/weights/lora-medvqa \
    --checkpoint-dir /kaggle/working/lora_training_run
```

## 7. Sanity-check the trained adapter
`model_loader.py` picks up `weights/lora-medvqa` automatically since that's `config.LORA_PATH` -- as long as we point it at the Kaggle output location first.

In [ ]:
import importlib
from src import config

config.LORA_PATH = "/kaggle/working/weights/lora-medvqa"

!python src/evaluate_omnimed.py \
    --data-root "{DATA_ROOT}" \
    --dataset-names ACRIMA \
    --max-samples 20

## 8. Retrieve your adapter
Click **Save Version** (or use the Output panel). Everything under `/kaggle/working/weights/lora-medvqa/` (just `adapter_config.json` + `adapter_model.safetensors`, tens of MB) becomes downloadable. Pull it into your submission repo at `weights/lora-medvqa/`, matching `config.LORA_PATH` exactly.

---
## 9. (Separate concern) Produce a smaller, pre-quantized base model for your SUBMISSION

**Read this before running.** Enabling `config.QUANTIZATION_MODE` only changes how the model is loaded INTO MEMORY -- it does NOT shrink the ~9GB fp16 weight files on disk by itself. To actually get a smaller submission, you must load the model once with quantization on and explicitly re-save it.

**Important, not just theoretical**: I verified quantized *loading* works correctly (~2.9GB VRAM vs 8.3GB fp16, confirmed via real parameter/dtype inspection). I could **not** fully verify the *save-to-disk* step locally -- my dev machine hit unrelated disk/memory limits partway through. Official HF docs describe `save_pretrained()` on a bnb-quantized model as supported (contingent on having the *latest* bitsandbytes), but at least one other source warns it can sometimes save the original weights back out instead, requiring re-quantization on every load. **Run this cell EARLY -- ideally before Section 6's fine-tuning run -- specifically so you still have time left for Plan B if the size check below fails.**

This is otherwise unrelated to the fine-tuning run above -- run it any time, as long as `config.MODEL_PATH` still points at the full fp16 weights first.

In [ ]:
# Upgrade first -- the HF docs specifically call out needing the latest
# bitsandbytes for quantized save/reload to work correctly.
!pip install -q -U bitsandbytes

In [ ]:
**If the size check passed** (`disk_gb` well under the original ~8.3GB, ideally ~2.5-3GB for 4-bit / ~4.5-5GB for 8-bit), for your submission:
1. Use `{QUANTIZED_OUTPUT_DIR}`'s contents as your `weights/qwen3-vl-4b-instruct/` in the submission zip (instead of the original fp16 folder).
2. In `src/config.py`, set `QUANTIZATION_MODE = "4bit"` (or `"8bit"`, matching what you ran above) -- belt-and-suspenders alongside the quantization_config already embedded in the saved checkpoint.
3. In `requirements.txt`, **uncomment** the `bitsandbytes>=0.43.0` line -- the evaluator's fresh environment needs it installed via `pip install -r requirements.txt`, or the submission will crash at model-load time.
4. Re-run `eval.py` against the dev set once with this quantized checkpoint before submitting, to confirm accuracy didn't degrade unacceptably and nothing crashes.

**If the size check failed** (checkpoint close to the original ~8.3GB): don't spend more time debugging bitsandbytes serialization under deadline pressure. In order of how much time each costs you:
1. Try `QUANT_MODE = "8bit"` instead -- 8-bit serialization support has historically been more mature/reliable than 4-bit in some bitsandbytes/transformers version combinations. Re-run this cell.
2. If that also fails, search Hugging Face for an already-published pre-quantized checkpoint of this exact model (search `Qwen3-VL-4B-Instruct` + `bnb-4bit`/`AWQ`/`GPTQ`) -- if one exists, download and use it directly instead of re-quantizing yourself, skipping this whole cell.
3. As a last resort: submit with the full fp16 weights and make the 10GB limit work by trimming everything else instead (a lean report with fewer embedded images, a smaller presentation file, no unused checkpoints/logs in the zip) -- a working unquantized submission beats a broken quantized one.

Confirm the size printed above, then for your submission:
1. Use `{QUANTIZED_OUTPUT_DIR}`'s contents as your `weights/qwen3-vl-4b-instruct/` in the submission zip (instead of the original fp16 folder).
2. In `src/config.py`, set `QUANTIZATION_MODE = "4bit"` (or `"8bit"`, matching what you ran above) -- belt-and-suspenders alongside the quantization_config already embedded in the saved checkpoint.
3. In `requirements.txt`, **uncomment** the `bitsandbytes>=0.43.0` line -- the evaluator's fresh environment needs it installed via `pip install -r requirements.txt`, or the submission will crash at model-load time.
4. Re-run `eval.py` against the dev set once with this quantized checkpoint before submitting, to confirm accuracy didn't degrade unacceptably and nothing crashes.